##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [ ]:
# place your code here
!pip install pinecone
!pip install python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


From 'pinecone' library import Pinecone, ServerlessSpec.

In [2]:
# place your code here

from pinecone import Pinecone, ServerlessSpec, SchemaBuilder

On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [7]:
# place your code here
import os, time
from dotenv import load_dotenv
load_dotenv()

pc = Pinecone(api_key= os.environ["PINECONE_API_KEY"])

schema = (
    SchemaBuilder()
    .add_string_field("chunk_text", full_text_search={"language": "en"}, embed={})
    .build()
)

index_name = "semantic-search-demo"
if not pc.indexes.exists(name=index_name):
    pc.indexes.create(name=index_name, schema=schema)
while not pc.indexes.describe(name=index_name).status.ready:
    time.sleep(2)
index = pc.index(name=index_name)

print("Index created")

data = [
    {"_id": "1", "chunk_text": "The Pacific Ocean is the largest and deepest ocean on Earth.", "category": "geography"},
    {"_id": "2", "chunk_text": "Mount Everest is the highest mountain above sea level.", "category": "geography"},
    {"_id": "3", "chunk_text": "Python is a popular programming language for data analysis.", "category": "technology"},
    {"_id": "4", "chunk_text": "Vector databases store embeddings for fast similarity search.", "category": "technology"},
    {"_id": "5", "chunk_text": "Error DB5021: the database connection was refused by the server.", "category": "technology"},
    {"_id": "6", "chunk_text": "Regular exercise improves heart health and reduces stress.", "category": "health"},
    {"_id": "7", "chunk_text": "A balanced diet includes vegetables, fruits and whole grains.", "category": "health"},
    {"_id": "8", "chunk_text": "The Roman Empire fell in the year 476 after centuries of rule.", "category": "history"},
    {"_id": "9", "chunk_text": "The printing press was invented by Johannes Gutenberg around 1440.", "category": "history"},
    {"_id": "10", "chunk_text": "Riga is the capital of Latvia and sits on the Baltic Sea.", "category": "geography"},
]

index.documents.upsert(
    namespace="__default__",
    documents = data,
)
time.sleep(10)

print("data inserted")

query = "Name the capital of Latvia?"

meaning = index.documents.search(
    namespace="__default__",
    top_k=3,
    score_by=[{"type": "embed", "fields": ["chunk_text"], "query": query}],
    include_fields=["chunk_text"],
)


# perform a search by exact terms
terms = index.documents.search(
    namespace="__default__",
    top_k=3,
    score_by=[{"type": "text", "fields": ["chunk_text"], "query": query}],
    include_fields=["chunk_text"],
)

Index created
data inserted


Print search results.

In [9]:
# place your code here
print("meaning search")
for m in meaning.matches:
    print(f"ID: {m.id}")
    print(f"Score: {m.score}")
    print(f"Text: {m.chunk_text}")



meaning search
ID: 10
Score: 0.5456492
Text: Riga is the capital of Latvia and sits on the Baltic Sea.
ID: 9
Score: 0.018738322
Text: The printing press was invented by Johannes Gutenberg around 1440.
ID: 6
Score: 0.018048368
Text: Regular exercise improves heart health and reduces stress.


In [10]:
print("exact term search")
for m in terms.matches:
    print(f"ID: {m.id}")
    print(f"Score: {m.score}")
    print(f"Text: {m.chunk_text}")

exact term search
ID: 10
Score: 5.6870866
Text: Riga is the capital of Latvia and sits on the Baltic Sea.
ID: 8
Score: 2.0374014
Text: The Roman Empire fell in the year 476 after centuries of rule.
ID: 5
Score: 0.71924967
Text: Error DB5021: the database connection was refused by the server.
